# Build a Streaming Chatbot, Step by Step

In Level 1, you sent a message and waited for the **entire** response before seeing anything. Streaming changes that — tokens appear one by one as the AI generates them, creating a natural "typing" effect.

**What's new compared to Level 1:**
- `stream=True` in the API call
- Iterating over an event stream instead of reading a single response
- Two event types: `response.output_text.delta` (a new chunk of text) and `response.completed` (all done)

**Prerequisites:** Complete the [Level 1 notebook](../level-1-chat/build-a-chatbot.ipynb) first, or at least have your `.env` file configured.

In [ ]:
# Install required packages into the Jupyter kernel
# (only needed once — safe to re-run)
%pip install openai python-dotenv

## Setup (Quick)

The setup is identical to Level 1 — load your `.env`, create an OpenAI client pointed at Azure. Since you've done this before, it's all in one cell.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

# Load .env from the project root (one level up from this notebook)
env_path = Path.cwd().parent / ".env"
if not env_path.exists():
    env_path = Path.cwd() / ".env"  # fallback: same directory
load_dotenv(env_path)

ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
MODEL = os.getenv("MODEL_NAME", "gpt-5.2")
REASONING_EFFORT = os.getenv("REASONING_EFFORT", "low")
INSTRUCTIONS = os.getenv(
    "CHATBOT_INSTRUCTIONS",
    "You are a helpful assistant. Be concise and friendly.",
)

if not ENDPOINT or not API_KEY:
    raise RuntimeError(
        f"Missing AZURE_OPENAI_ENDPOINT or AZURE_OPENAI_API_KEY.\n"
        f"Looked for .env at: {env_path}\n"
        f"Copy .env.example to .env and fill in your values."
    )

client = OpenAI(
    base_url=ENDPOINT,
    api_key=API_KEY,
    max_retries=10,
)

print(f"Model: {MODEL}  |  Reasoning effort: {REASONING_EFFORT}")
print("Client ready!")

## Non-Streaming vs Streaming

In Level 1, every API call worked like this:

```
Non-streaming:  wait... wait... wait... [full response appears all at once]
```

With streaming, tokens arrive **one by one** as the model generates them:

```
Streaming:  "Hello" ... " world" ... "!" ... [done]
```

The end result is the same text, but the user experience is completely different. Streaming gives a natural "typing" effect — the user starts reading immediately instead of staring at a blank screen.

## How Streaming Works

Add `stream=True` to the API call. Instead of getting back a response object, you get an **iterable of events**. Two event types matter:

| Event type | What it means | Key attribute |
|---|---|---|
| `response.output_text.delta` | A chunk of text just arrived | `event.delta` — the new text fragment |
| `response.completed` | The full response is done | `event.response.id` — the response ID (for chaining turns) |

You iterate over the stream with a `for` loop, check `event.type`, and handle each event accordingly.

In [ ]:
import sys

stream = client.responses.create(
    model=MODEL,
    input="Write a short poem about coding.",
    instructions=INSTRUCTIONS,
    reasoning={"effort": REASONING_EFFORT},
    stream=True,
)

usage = None
print("Assistant: ", end="")
for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
    elif event.type == "response.completed":
        usage = event.response.usage
print()
print()
if usage:
    print(f"Tokens used: {usage.input_tokens} in, {usage.output_tokens} out, {usage.total_tokens} total")

## Comparing Non-Streaming and Streaming

Let's see the difference side by side. The next cell sends the **same message** both ways so you can compare the timing.

In [ ]:
import time

message = "Explain what an API is in 2-3 sentences."

# --- Non-streaming (like Level 1) ---
print("=== Non-Streaming ===")
start = time.time()
response = client.responses.create(
    model=MODEL,
    input=message,
    instructions=INSTRUCTIONS,
    reasoning={"effort": REASONING_EFFORT},
)
elapsed = time.time() - start
print(f"Assistant: {response.output_text}")
print(f"(Waited {elapsed:.1f}s for the complete response)\n")

# --- Streaming ---
print("=== Streaming ===")
start = time.time()
first_token_time = None
stream = client.responses.create(
    model=MODEL,
    input=message,
    instructions=INSTRUCTIONS,
    reasoning={"effort": REASONING_EFFORT},
    stream=True,
)

print("Assistant: ", end="")
for event in stream:
    if event.type == "response.output_text.delta":
        if first_token_time is None:
            first_token_time = time.time() - start
        print(event.delta, end="", flush=True)
elapsed = time.time() - start
print(f"\n(First token in {first_token_time:.1f}s, complete in {elapsed:.1f}s)")

## Streaming with Conversation Chaining

Conversation chaining works the same way as Level 1 — pass `previous_response_id` to continue the conversation. The only difference is **where** you get the response ID: from the `response.completed` event instead of the response object.

```python
elif event.type == "response.completed":
    previous_response_id = event.response.id  # <-- grab it here
```

In [ ]:
# --- Turn 1 ---
print("You: What is recursion?\n")
print("Assistant: ", end="")

previous_response_id = None

stream = client.responses.create(
    model=MODEL,
    input="What is recursion?",
    instructions=INSTRUCTIONS,
    reasoning={"effort": REASONING_EFFORT},
    stream=True,
)

for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
    elif event.type == "response.completed":
        previous_response_id = event.response.id

print("\n")

# --- Turn 2 ---
print("You: Can you give me a simple example?\n")
print("Assistant: ", end="")

stream = client.responses.create(
    model=MODEL,
    input="Can you give me a simple example?",
    instructions=INSTRUCTIONS,
    reasoning={"effort": REASONING_EFFORT},
    previous_response_id=previous_response_id,
    stream=True,
)

for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
    elif event.type == "response.completed":
        previous_response_id = event.response.id

print()

## You Built a Streaming Chatbot!

Here's what you learned:

1. **Add `stream=True`** to the API call to get an event stream instead of a complete response
2. **Handle `response.output_text.delta`** to print text chunks as they arrive
3. **Use `response.completed`** to grab the response ID for conversation chaining
4. **Streaming gives faster time-to-first-token** — the user starts reading immediately

### Try it out

Run the full streaming chatbot from a terminal:

```bash
cd level-1-chat-streaming/python
python chat.py
```

### What's next?

- **[Level 2 — File Upload](../level-2-files/)** — attach images and documents to your messages
- **[Level 3 — Web UI](../level-3-web/)** — build a browser-based chat interface with streaming
- **[PRDs](../PRDs/)** — workshop exercises to extend your chatbot with Claude Code